### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:
1. A schema including the name of the tool, a description and/or argument definitions(after a JSON schema).
2. A function or coroutine to execute

In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import langchain
load_dotenv()

os.environ["Groq_API_key"] = os.getenv("Groq_API_key")
llm = init_chat_model(
    model="llama-3.3-70b-versatile",
    model_provider="groq",
    api_key=os.getenv("GROQ_API_KEY"),
)
response = llm.invoke("Why do parrots talk?")
response

AIMessage(content="The ability of parrots to talk, also known as vocal mimicry, is a complex phenomenon that has fascinated humans for centuries. While we can't directly ask a parrot why it talks, research and observations have provided some insights into the possible reasons behind this behavior.\n\n1. **Communication and social bonding**: In the wild, parrots use vocalizations to communicate with each other, forming strong social bonds and conveying information about food, predators, and potential mates. Talking may be an extension of this natural behavior, allowing parrots to interact with their human caregivers and establish a connection.\n2. **Mimicry and learning**: Parrots are renowned for their exceptional ability to mimic sounds, including human speech. This skill is thought to be an adaptation for learning and remembering complex vocalizations, which helps them to communicate effectively with other parrots. By mimicking human speech, parrots may be exercising their vocal lear

In [3]:
from langchain.tools import tool
@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    # In a real implementation, you would call a weather API here
    return f"The current weather in {location} is sunny with a temperature of 25°C."


model_with_tools = llm.bind_tools([get_current_weather])

In [4]:
response_with_tools = model_with_tools.invoke("What is the current weather in Ahmedabad?")
print(response_with_tools.content)
for tool_call in response_with_tools.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")



Tool: get_current_weather
Args: {'location': 'Ahmedabad'}


### Tool execution loop

In [7]:
#Step 1: Model generates a tool call
messages = [
    {"role": "user", "content": "What's the weather in Ahmedabad?"},]
ai_msg = llm.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect the results
for tool_call in ai_msg.tool_calls:
    tool_result = get_current_weather(**tool_call["args"])
    messages.append(tool_result)

#Step 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Ahmedabad?"},
 AIMessage(content="I'm an AI, I don't have real-time access to current weather conditions. However, I can suggest some ways for you to find out the current weather in Ahmedabad:\n\n1. Check online weather websites: You can visit websites like AccuWeather, Weather.com, or the India Meteorological Department (IMD) website to get the current weather conditions in Ahmedabad.\n2. Use a weather app: You can download a weather app on your smartphone, such as Dark Sky or Weather Underground, to get the current weather conditions in Ahmedabad.\n3. Check social media: You can also check social media platforms like Twitter or Facebook for updates on the current weather conditions in Ahmedabad.\n\nAhmedabad has a hot and dry climate, with very little rainfall throughout the year. The city experiences a hot summer from March to May, a monsoon season from June to September, and a mild winter from October to February.\n\nIf you're pla